Here is the code where you put in H, M, G, model and measurement

In [49]:
#| default_exp prediction_module

In [37]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

Normalize is in data module but i dont know how to import

This function is normalizing the persons weight and height

In [38]:
#| export
def normalize_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    normalize(person, 'weightkg')
    normalize(person, 'stature')
    return person

This functions gets the pickled model

In [39]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the normalized person to predict the measurement

In [40]:
#| export
def predict_from_model(kindofmodel:str, measurement:str, w, h, g):
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    person = normalize_person(w, h, g)
    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    elif kindofmodel=='bambi':
        model = model_bmb(measurement)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    elif kindofmodel=='bambi_ml_gc':
        train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
        model = component_model(measurement,train)
        person['Component']='Army Reserve' #'Regular Army'#'Army National Guard'#'Army Reserve'
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    else:
        return 'wrong model name'

In [41]:
#| export
def add_to_df(df, measurement, pred):
    df[measurement]=pred
    return df

I dont know if we need to make it to csv?

In [100]:
#| export
def make_csv(data, name):
    data.to_csv(f'{name}.csv', index=False)

Here are all predictions made, the measurements are in a list

In [43]:
#| export
def make_predictions(kindofmodel:str, measurements:list, w, h, g):
    output=pd.DataFrame()
    for m in measurements:
        pred = predict_from_model(kindofmodel, m, w, h, g)
        add_to_df(output, m, pred)
    return output

In [56]:
make_predictions('bambi',['abdominalextensiondepthsitting','acromialheight'], 80, 1700, 1)

,abdominalextensiondepthsitting,acromialheight
0,250.066345,1390.5659


In [81]:
make_predictions('xgboost',['abdominalextensiondepthsitting','acromialheight','neckcircumference'], 81.5, 1776, 1)

,abdominalextensiondepthsitting,acromialheight,neckcircumference
0,252.47998,1457.88855,384.442688


In [ ]:
test=pd.read_csv('../data/processed/ANSURIItest.csv')
test_preds_all=pd.DataFrame()
variables = test.columns[:6]

for i in range(10):
    testrad=test.iloc[i]
    test_pred_xgb = make_predictions('xgboost',variables, testrad['weightkg'], testrad['stature'], testrad['Gender'])
    test_preds_all = pd.concat([test_preds_all, test_pred_xgb], ignore_index=True)

test_preds_all

,abdominalextensiondepthsitting,acromialheight,acromionradialelength,anklecircumference,axillaheight,balloffootcircumference
0,238.612579,1493.657349,349.952972,229.757187,1385.080933,257.072205
1,218.673401,1532.893921,359.418060,220.101547,1426.385986,248.927032
2,262.359650,1423.951050,333.484161,230.973175,1320.708008,238.043793
3,233.813248,1479.071533,347.110718,227.849136,1362.561401,240.715683
4,233.233109,1488.230225,345.965454,225.523285,1374.382812,254.071747
5,201.262268,1370.299927,316.221436,211.792969,1277.242554,230.744934
6,202.237198,1291.821777,296.726227,203.924377,1200.723999,219.257721
7,300.203949,1370.301636,315.716187,236.212463,1256.608887,251.706039
8,219.368149,1443.904053,339.410522,220.281067,1340.671265,246.298782
9,228.784958,1351.073486,315.280334,216.972946,1250.770386,239.481750


In [74]:
test_preds_all

,abdominalextensiondepthsitting,acromialheight,acromionradialelength,anklecircumference,axillaheight,balloffootcircumference,balloffootlength,biacromialbreadth,bicepscircumferenceflexed,bicristalbreadth,...,trochanterionheight,verticaltrunkcircumferenceusa,waistbacklength,waistbreadth,waistcircumference,waistdepth,waistfrontlengthsitting,waistheightomphalion,wristcircumference,wristheight
0,180.804596,1177.109619,266.849213,184.875153,1091.722534,207.045319,172.057755,355.735321,257.657928,235.294876,...,723.026672,1350.754639,412.230835,242.327148,682.864929,165.393021,315.517303,859.906738,151.100189,709.287598
1,180.804596,1177.109619,266.849213,184.875153,1091.722534,207.045319,172.057755,355.735321,257.657928,235.294876,...,723.026672,1350.754639,412.230835,242.327148,682.864929,165.393021,315.517303,859.906738,151.100189,709.287598
2,182.089081,1179.587280,267.723083,194.775345,1093.539551,205.790192,164.496201,329.111603,248.179428,238.665771,...,727.053833,1388.528564,379.755707,240.430695,685.492371,165.680664,319.404694,861.543640,135.969559,715.745789


In [101]:
make_csv(test_preds_all,'../output/anthro_results/predict_test_xgboost8')

In [50]:
import nbdev; nbdev.nbdev_export()